In [1]:
# ==========================================================
# ALGORITHME ID3 FROM SCRATCH EN PYTHON
# ==========================================================
#
# ID3 = Iterative Dichotomiser 3
#
# C'est un algorithme utilisé pour construire
# un arbre de décision.
#
# ----------------------------------------------------------
# OBJECTIF :
# ----------------------------------------------------------
#
# Trouver automatiquement les meilleures questions
# à poser pour classifier les données.
#
# Exemple :
#
# "La météo est-elle Soleil ?"
# "L'humidité est-elle Haute ?"
#
# afin de prédire :
#
# -> Jouer = Oui / Non
#
# ----------------------------------------------------------
# ID3 UTILISE :
# ----------------------------------------------------------
#
# 1. ENTROPIE
#    -> mesure le désordre
#
# 2. GAIN D'INFORMATION
#    -> mesure la qualité d'une séparation
#
# ----------------------------------------------------------
# PLUS LE GAIN EST GRAND :
# ----------------------------------------------------------
#
# -> meilleure est la feature
#
# ==========================================================

# ==========================================================
# IMPORTATION DES LIBRARIES
# ==========================================================

# pandas :
# utilisé pour manipuler les tableaux de données
import pandas as pd

# numpy :
# utilisé pour les calculs mathématiques
import numpy as np

# Counter :
# utilisé pour compter les occurrences
# Exemple :
# Oui = 6
# Non = 4
from collections import Counter

# ==========================================================
# 1. CRÉATION DU DATASET
# ==========================================================

# ----------------------------------------------------------
# Nous allons créer un dataset simple :
#
# Objectif :
# prédire si on joue au tennis
#
# Variable cible :
# Jouer = Oui / Non
#
# Features :
# Meteo
# Temperature
# Humidite
# Vent
# ----------------------------------------------------------

data = {

    # ------------------------------------------------------
    # FEATURE 1 : MÉTÉO
    # ------------------------------------------------------
    #
    # Soleil
    # Nuageux
    # Pluie
    #
    # Cette colonne représente la météo
    #
    # ------------------------------------------------------

    'Meteo': [

        'Soleil',
        'Soleil',
        'Nuageux',
        'Pluie',
        'Pluie',
        'Pluie',
        'Nuageux',
        'Soleil',
        'Soleil',
        'Pluie'
    ],

    # ------------------------------------------------------
    # FEATURE 2 : TEMPÉRATURE
    # ------------------------------------------------------
    #
    # Chaud
    # Moyen
    # Froid
    #
    # ------------------------------------------------------

    'Temperature': [

        'Chaud',
        'Chaud',
        'Chaud',
        'Moyen',
        'Froid',
        'Froid',
        'Froid',
        'Moyen',
        'Froid',
        'Moyen'
    ],

    # ------------------------------------------------------
    # FEATURE 3 : HUMIDITÉ
    # ------------------------------------------------------

    'Humidite': [

        'Haute',
        'Haute',
        'Haute',
        'Haute',
        'Normale',
        'Normale',
        'Normale',
        'Haute',
        'Normale',
        'Normale'
    ],

    # ------------------------------------------------------
    # FEATURE 4 : VENT
    # ------------------------------------------------------

    'Vent': [

        'Faible',
        'Fort',
        'Faible',
        'Faible',
        'Faible',
        'Fort',
        'Fort',
        'Faible',
        'Faible',
        'Faible'
    ],

    # ------------------------------------------------------
    # VARIABLE CIBLE
    # ------------------------------------------------------
    #
    # Oui  -> jouer
    # Non  -> ne pas jouer
    #
    # ------------------------------------------------------

    'Jouer': [

        'Non',
        'Non',
        'Oui',
        'Oui',
        'Oui',
        'Non',
        'Oui',
        'Non',
        'Oui',
        'Oui'
    ]
}

# ----------------------------------------------------------
# Transformer dictionnaire -> DataFrame
# ----------------------------------------------------------

df = pd.DataFrame(data)

# ----------------------------------------------------------
# Affichage du dataset
# ----------------------------------------------------------

print("===== DATASET =====")

print(df)

# ==========================================================
# 2. FONCTION ENTROPIE
# ==========================================================

def entropy(y):

    """
    ------------------------------------------------------
    ENTROPIE
    ------------------------------------------------------

    L'entropie mesure le niveau de désordre.

    ------------------------------------------------------
    CAS 1 :
    ------------------------------------------------------

    Oui Oui Oui Oui

    -> entropie faible
    -> données pures

    ------------------------------------------------------
    CAS 2 :
    ------------------------------------------------------

    Oui Non Oui Non

    -> entropie élevée
    -> données mélangées

    ------------------------------------------------------
    FORMULE :
    ------------------------------------------------------

    Entropy(S) = - Σ p(x) log2(p(x))

    ------------------------------------------------------
    p(x) :
    ------------------------------------------------------

    probabilité d'une classe

    Exemple :
    Oui = 6/10

    ------------------------------------------------------
    """

    # ------------------------------------------------------
    # Compter les occurrences des classes
    # ------------------------------------------------------
    #
    # Exemple :
    #
    # Oui = 6
    # Non = 4
    #
    # ------------------------------------------------------

    counts = Counter(y)

    # ------------------------------------------------------
    # Nombre total d'exemples
    # ------------------------------------------------------

    total = len(y)

    # ------------------------------------------------------
    # Variable entropie
    # ------------------------------------------------------

    ent = 0

    # ------------------------------------------------------
    # Parcourir chaque classe
    # ------------------------------------------------------

    for count in counts.values():

        # --------------------------------------------------
        # Calcul probabilité
        # --------------------------------------------------
        #
        # Exemple :
        #
        # Oui = 6/10 = 0.6
        #
        # --------------------------------------------------

        p = count / total

        # --------------------------------------------------
        # Formule entropie
        # --------------------------------------------------

        ent -= p * np.log2(p)

    # ------------------------------------------------------
    # Retourner entropie finale
    # ------------------------------------------------------

    return ent

# ==========================================================
# 3. GAIN D'INFORMATION
# ==========================================================

def information_gain(data, feature, target):

    """
    ------------------------------------------------------
    GAIN D'INFORMATION
    ------------------------------------------------------

    Le gain mesure :

    "combien une feature réduit le désordre"

    ------------------------------------------------------
    PLUS LE GAIN EST GRAND :
    ------------------------------------------------------

    -> meilleure séparation

    -> meilleure feature

    ------------------------------------------------------
    EXEMPLE :
    ------------------------------------------------------

    Si "Meteo" sépare très bien les classes,
    alors son gain sera élevé.
    """

    # ------------------------------------------------------
    # ÉTAPE 1 :
    # Calcul entropie totale
    # ------------------------------------------------------

    total_entropy = entropy(data[target])

    # ------------------------------------------------------
    # Récupérer valeurs uniques
    # ------------------------------------------------------
    #
    # Exemple :
    #
    # Meteo :
    # Soleil
    # Pluie
    # Nuageux
    #
    # ------------------------------------------------------

    values = data[feature].unique()

    # ------------------------------------------------------
    # Entropie pondérée
    # ------------------------------------------------------

    weighted_entropy = 0

    # ------------------------------------------------------
    # Parcourir chaque valeur
    # ------------------------------------------------------

    for value in values:

        # --------------------------------------------------
        # Créer sous-ensemble
        # --------------------------------------------------
        #
        # Exemple :
        #
        # Meteo = Soleil
        #
        # --------------------------------------------------

        subset = data[data[feature] == value]

        # --------------------------------------------------
        # Calcul entropie sous-ensemble
        # --------------------------------------------------

        subset_entropy = entropy(subset[target])

        # --------------------------------------------------
        # Calcul poids
        # --------------------------------------------------
        #
        # poids =
        # taille sous-ensemble / taille totale
        #
        # --------------------------------------------------

        weight = len(subset) / len(data)

        # --------------------------------------------------
        # Ajouter entropie pondérée
        # --------------------------------------------------

        weighted_entropy += weight * subset_entropy

    # ------------------------------------------------------
    # FORMULE GAIN
    # ------------------------------------------------------
    #
    # Gain =
    #
    # entropie totale
    # -
    # entropie pondérée
    #
    # ------------------------------------------------------

    gain = total_entropy - weighted_entropy

    return gain

# ==========================================================
# 4. TROUVER MEILLEURE FEATURE
# ==========================================================

def best_feature(data, features, target):

    """
    ------------------------------------------------------
    Cette fonction teste toutes les features
    et choisit celle qui a :
    ------------------------------------------------------

    -> le plus grand gain

    ------------------------------------------------------
    Exemple :
    ------------------------------------------------------

    Meteo       -> 0.42
    Humidite    -> 0.18
    Vent        -> 0.09

    => Meteo sera choisie
    """

    # ------------------------------------------------------
    # Dictionnaire gains
    # ------------------------------------------------------

    gains = {}

    # ------------------------------------------------------
    # Calcul gain de chaque feature
    # ------------------------------------------------------

    for feature in features:

        gains[feature] = information_gain(
            data,
            feature,
            target
        )

    # ------------------------------------------------------
    # Affichage gains
    # ------------------------------------------------------

    print("\n===== GAINS D'INFORMATION =====")

    for key, value in gains.items():

        print(f"{key} : {value:.4f}")

    # ------------------------------------------------------
    # Retourner meilleure feature
    # ------------------------------------------------------

    return max(gains, key=gains.get)

# ==========================================================
# 5. CONSTRUCTION RÉCURSIVE DE L'ARBRE
# ==========================================================

def id3(data, features, target):

    """
    ------------------------------------------------------
    Fonction principale de ID3
    ------------------------------------------------------

    Elle construit l'arbre récursivement.

    ------------------------------------------------------
    PRINCIPE :
    ------------------------------------------------------

    1. choisir meilleure feature

    2. créer branches

    3. recommencer sur chaque branche

    jusqu'à obtenir des feuilles finales
    """

    # ------------------------------------------------------
    # Classes présentes
    # ------------------------------------------------------

    labels = data[target]

    # ======================================================
    # CAS D'ARRÊT 1
    # ======================================================
    #
    # Toutes les classes identiques
    #
    # Exemple :
    #
    # Oui Oui Oui
    #
    # => feuille finale
    #
    # ======================================================

    if len(np.unique(labels)) == 1:

        return labels.iloc[0]

    # ======================================================
    # CAS D'ARRÊT 2
    # ======================================================
    #
    # Plus de features disponibles
    #
    # ======================================================

    if len(features) == 0:

        # retourner classe majoritaire

        return labels.mode()[0]

    # ======================================================
    # ÉTAPE 1 :
    # choisir meilleure feature
    # ======================================================

    best = best_feature(
        data,
        features,
        target
    )

    # ======================================================
    # Création arbre
    # ======================================================

    tree = {best: {}}

    # ======================================================
    # Valeurs possibles feature
    # ======================================================

    values = data[best].unique()

    # ======================================================
    # Création branches
    # ======================================================

    for value in values:

        # --------------------------------------------------
        # Sous-ensemble correspondant
        # --------------------------------------------------

        subset = data[data[best] == value]

        # --------------------------------------------------
        # Supprimer feature déjà utilisée
        # --------------------------------------------------

        remaining_features = [

            f for f in features

            if f != best
        ]

        # --------------------------------------------------
        # APPEL RÉCURSIF
        # --------------------------------------------------
        #
        # construire sous-arbre
        #
        # --------------------------------------------------

        subtree = id3(
            subset,
            remaining_features,
            target
        )

        # --------------------------------------------------
        # Ajouter sous-arbre
        # --------------------------------------------------

        tree[best][value] = subtree

    # ------------------------------------------------------
    # Retourner arbre final
    # ------------------------------------------------------

    return tree

# ==========================================================
# 6. ENTRAINEMENT DU MODÈLE
# ==========================================================

# ----------------------------------------------------------
# Features explicatives
# ----------------------------------------------------------

features = [

    'Meteo',
    'Temperature',
    'Humidite',
    'Vent'
]

# ----------------------------------------------------------
# Variable cible
# ----------------------------------------------------------

target = 'Jouer'

# ----------------------------------------------------------
# Construction arbre
# ----------------------------------------------------------

tree = id3(
    df,
    features,
    target
)

# ==========================================================
# 7. AFFICHAGE ARBRE FINAL
# ==========================================================

print("\n===== ARBRE DE DÉCISION =====")

print(tree)

# ==========================================================
# 8. PRÉDICTION
# ==========================================================

def predict(tree, sample):

    """
    ------------------------------------------------------
    Cette fonction fait une prédiction
    avec l'arbre construit.
    ------------------------------------------------------
    """

    # ------------------------------------------------------
    # Récupérer racine
    # ------------------------------------------------------

    root = list(tree.keys())[0]

    # ------------------------------------------------------
    # Valeur correspondante dans exemple
    # ------------------------------------------------------

    value = sample[root]

    # ------------------------------------------------------
    # Aller dans branche correspondante
    # ------------------------------------------------------

    subtree = tree[root][value]

    # ------------------------------------------------------
    # Si feuille
    # ------------------------------------------------------

    if not isinstance(subtree, dict):

        return subtree

    # ------------------------------------------------------
    # Sinon continuer récursivement
    # ------------------------------------------------------

    return predict(subtree, sample)

# ==========================================================
# 9. TEST
# ==========================================================

# ----------------------------------------------------------
# Nouvel exemple
# ----------------------------------------------------------

sample = {

    'Meteo': 'Soleil',

    'Temperature': 'Froid',

    'Humidite': 'Normale',

    'Vent': 'Faible'
}

# ----------------------------------------------------------
# Faire prédiction
# ----------------------------------------------------------

prediction = predict(tree, sample)

# ==========================================================
# 10. AFFICHAGE RÉSULTAT
# ==========================================================

print("\n===== PRÉDICTION =====")

print("Exemple :", sample)

print("Classe prédite :", prediction)

===== DATASET =====
     Meteo Temperature Humidite    Vent Jouer
0   Soleil       Chaud    Haute  Faible   Non
1   Soleil       Chaud    Haute    Fort   Non
2  Nuageux       Chaud    Haute  Faible   Oui
3    Pluie       Moyen    Haute  Faible   Oui
4    Pluie       Froid  Normale  Faible   Oui
5    Pluie       Froid  Normale    Fort   Non
6  Nuageux       Froid  Normale    Fort   Oui
7   Soleil       Moyen    Haute  Faible   Non
8   Soleil       Froid  Normale  Faible   Oui
9    Pluie       Moyen  Normale  Faible   Oui

===== GAINS D'INFORMATION =====
Meteo : 0.3219
Temperature : 0.0955
Humidite : 0.1245
Vent : 0.0913

===== GAINS D'INFORMATION =====
Temperature : 0.8113
Humidite : 0.8113
Vent : 0.1226

===== GAINS D'INFORMATION =====
Temperature : 0.3113
Humidite : 0.1226
Vent : 0.8113

===== ARBRE DE DÉCISION =====
{'Meteo': {'Soleil': {'Temperature': {'Chaud': 'Non', 'Moyen': 'Non', 'Froid': 'Oui'}}, 'Nuageux': 'Oui', 'Pluie': {'Vent': {'Faible': 'Oui', 'Fort': 'Non'}}}}

===== PRÉ